In [3]:
!ls -lh /content

total 44M
-rw-r--r-- 1 root root  44M Sep 25 18:13 online_retail_II.xlsx
drwxr-xr-x 1 root root 4.0K Sep 16 13:26 sample_data


## 1. Import Libraries

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [7]:
file_path = "/content/online_retail_II.xlsx"

excel_file = pd.ExcelFile(file_path)

excel_file.sheet_names

['Year 2009-2010', 'Year 2010-2011']

## 2. Load Raw Transaction Data

The source workbook contains two years of e-commerce transaction records.  
The two worksheets are loaded separately and later combined into a single analytical dataset.


In [8]:
df_2009 = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

df_2010 = pd.read_excel(
    file_path,
    sheet_name="Year 2010-2011"
)

In [9]:
print("2009-2010:", df_2009.shape)
print("2010-2011:", df_2010.shape)


2009-2010: (525461, 8)
2010-2011: (541910, 8)


In [10]:
df_2009.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [11]:
df_2010.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,"17,850.00",United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,"17,850.00",United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,"17,850.00",United Kingdom


In [13]:
df_2009["SourceYear"] = "2009-2010"
df_2010["SourceYear"] = "2010-2011"


## 3. Combine Annual Transaction Data

In [14]:
df = pd.concat(
    [df_2009, df_2010],
    ignore_index=True
)

df.shape


(1067371, 9)

In [15]:
df_raw = df.copy()


In [16]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'SourceYear'],
      dtype='object')

In [17]:
df = df.rename(columns={
    "Invoice": "InvoiceNo",
    "Customer ID": "CustomerID",
    "Price": "UnitPrice"
})

df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'SourceYear'],
      dtype='object')

## 4. Initial Data Quality Assessment

In [18]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   InvoiceNo    1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   UnitPrice    1067371 non-null  float64       
 6   CustomerID   824364 non-null   float64       
 7   Country      1067371 non-null  object        
 8   SourceYear   1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(5)
memory usage: 73.3+ MB


In [19]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

missing_summary

,Missing_Count,Missing_Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,243007,22.77
Country,0,0.00
SourceYear,0,0.00


In [20]:
duplicate_count = df.duplicated().sum()

print("Number of exact duplicate rows:", duplicate_count)

Number of exact duplicate rows: 12133


In [21]:
df[df.duplicated(keep=False)].sort_values(
    by=["InvoiceNo", "StockCode"]
).head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear
379,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,2009-2010
391,489517,21491,SET OF THREE VINTAGE GIFT WRAPS,1,2009-12-01 11:34:00,1.95,"16,329.00",United Kingdom,2009-2010
365,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
386,489517,21821,GLITTER STAR GARLAND WITH BELLS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
363,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
371,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
394,489517,21912,VINTAGE SNAKES & LADDERS,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
362,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
385,489517,21913,VINTAGE SEASIDE JIGSAW PUZZLES,1,2009-12-01 11:34:00,3.75,"16,329.00",United Kingdom,2009-2010
368,489517,22130,PARTY CONE CHRISTMAS DECORATION,6,2009-12-01 11:34:00,0.85,"16,329.00",United Kingdom,2009-2010


In [22]:
df[["Quantity", "UnitPrice"]].describe()

,Quantity,UnitPrice
count,"1,067,371.00","1,067,371.00"
mean,9.94,4.65
std,172.71,123.55
min,"-80,995.00","-53,594.36"
25%,1.00,1.25
50%,3.00,2.10
75%,10.00,4.15
max,"80,995.00","38,970.00"


In [23]:
negative_quantity = df[df["Quantity"] < 0]

print("Negative quantity rows:", len(negative_quantity))

negative_quantity.head(10)

Negative quantity rows: 22950


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,2009-2010
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,"16,321.00",Australia,2009-2010
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,"16,321.00",Australia,2009-2010
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,"16,321.00",Australia,2009-2010
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,2009-2010
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,2009-2010
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,2009-2010
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,"16,321.00",Australia,2009-2010
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,2009-2010
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,"17,592.00",United Kingdom,2009-2010


In [24]:
df["InvoiceNo"] = df["InvoiceNo"].astype(str)


In [25]:
df["IsCancelled"] = df["InvoiceNo"].str.startswith("C")


In [26]:
df["IsCancelled"].value_counts()

,count
IsCancelled,
False,1047877
True,19494


In [27]:
cancellation_summary = (
    df["IsCancelled"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

cancellation_summary

,proportion
IsCancelled,
False,98.17
True,1.83


In [28]:
df[df["IsCancelled"]].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,2009-2010,True
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,"16,321.00",Australia,2009-2010,True
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,"16,321.00",Australia,2009-2010,True
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,"16,321.00",Australia,2009-2010,True
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,2009-2010,True
183,C489449,21871,SAVE THE PLANET MUG,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,2009-2010,True
184,C489449,84946,ANTIQUE SILVER TEA GLASS ETCHED,-12,2009-12-01 10:33:00,1.25,"16,321.00",Australia,2009-2010,True
185,C489449,84970S,HANGING HEART ZINC T-LIGHT HOLDER,-24,2009-12-01 10:33:00,0.85,"16,321.00",Australia,2009-2010,True
186,C489449,22090,PAPER BUNTING RETRO SPOTS,-12,2009-12-01 10:33:00,2.95,"16,321.00",Australia,2009-2010,True
196,C489459,90200A,PURPLE SWEETHEART BRACELET,-3,2009-12-01 10:44:00,4.25,"17,592.00",United Kingdom,2009-2010,True


In [29]:
df.groupby("IsCancelled")["Quantity"].describe()

,count,mean,std,min,25%,50%,75%,max
IsCancelled,,,,,,,,
False,"1,047,877.00",10.59,135.28,"-9,600.00",1.00,3.00,10.00,"80,995.00"
True,"19,494.00",-25.19,805.10,"-80,995.00",-6.00,-2.00,-1.00,1.00


In [30]:
price_issues = df[df["UnitPrice"] <= 0]

print("Rows with zero or negative price:", len(price_issues))

price_issues.head(20)

Rows with zero or negative price: 6207


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,2009-2010,False
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,2009-2010,False
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom,2009-2010,False
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom,2009-2010,False
3162,489660,35956,lost,-1043,2009-12-01 17:43:00,0.00,NaN,United Kingdom,2009-2010,False
3168,489663,35605A,damages,-117,2009-12-01 18:02:00,0.00,NaN,United Kingdom,2009-2010,False
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom,2009-2010,False
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom,2009-2010,False


In [31]:
missing_customer = df[df["CustomerID"].isna()]

print("Rows without CustomerID:", len(missing_customer))

missing_customer.head(10)

Rows without CustomerID: 243007


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010,False
283,489463,71477,short,-240,2009-12-01 10:52:00,0.00,NaN,United Kingdom,2009-2010,False
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.00,NaN,United Kingdom,2009-2010,False
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,2009-2010,False
577,489525,85226C,BLUE PULL BACK RACING CAR,1,2009-12-01 11:49:00,0.55,NaN,United Kingdom,2009-2010,False
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,2009-12-01 11:49:00,0.85,NaN,United Kingdom,2009-2010,False
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2009-12-01 12:32:00,2.95,NaN,United Kingdom,2009-2010,False
1056,489548,22254,FELT TOADSTOOL LARGE,12,2009-12-01 12:32:00,1.25,NaN,United Kingdom,2009-2010,False
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2009-12-01 12:32:00,2.95,NaN,United Kingdom,2009-2010,False
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,2009-12-01 12:32:00,1.65,NaN,United Kingdom,2009-2010,False


In [32]:
missing_description = df[df["Description"].isna()]

print("Rows without Description:", len(missing_description))

missing_description.head(10)

Rows without Description: 4382


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.00,NaN,United Kingdom,2009-2010,False
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.00,NaN,United Kingdom,2009-2010,False
3161,489659,21350,NaN,230,2009-12-01 17:39:00,0.00,NaN,United Kingdom,2009-2010,False
3731,489781,84292,NaN,17,2009-12-02 11:45:00,0.00,NaN,United Kingdom,2009-2010,False
4296,489806,18010,NaN,-770,2009-12-02 12:42:00,0.00,NaN,United Kingdom,2009-2010,False
4566,489821,85049G,NaN,-240,2009-12-02 13:25:00,0.00,NaN,United Kingdom,2009-2010,False
6378,489882,35751C,NaN,12,2009-12-02 16:22:00,0.00,NaN,United Kingdom,2009-2010,False
6555,489898,79323G,NaN,954,2009-12-03 09:40:00,0.00,NaN,United Kingdom,2009-2010,False
6576,489901,21098,NaN,-200,2009-12-03 09:47:00,0.00,NaN,United Kingdom,2009-2010,False
6581,489903,21166,NaN,48,2009-12-03 09:57:00,0.00,NaN,United Kingdom,2009-2010,False


In [33]:
df["InvoiceDate"].dtype

dtype('<M8[ns]')

In [34]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce"
)

In [35]:
df["InvoiceDate"].isna().sum()

np.int64(0)

In [36]:
df["Year"] = df["InvoiceDate"].dt.year
df["Month"] = df["InvoiceDate"].dt.month
df["YearMonth"] = df["InvoiceDate"].dt.to_period("M").astype(str)
df["DayOfWeek"] = df["InvoiceDate"].dt.day_name()
df["Hour"] = df["InvoiceDate"].dt.hour

In [37]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled,Year,Month,YearMonth,DayOfWeek,Hour
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7


In [38]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

In [39]:
df[[
    "InvoiceNo",
    "Quantity",
    "UnitPrice",
    "Revenue"
]].head()

,InvoiceNo,Quantity,UnitPrice,Revenue
0,489434,12,6.95,83.40
1,489434,12,6.75,81.00
2,489434,12,6.75,81.00
3,489434,48,2.10,100.80
4,489434,24,1.25,30.00


In [40]:
sales = df[
    (~df["IsCancelled"]) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()

In [41]:
sales.shape

(1041670, 16)

In [42]:
sales_duplicates = sales.duplicated().sum()

print("Duplicate sales rows:", sales_duplicates)

Duplicate sales rows: 12061


In [43]:
sales = sales.drop_duplicates().copy()

In [44]:
sales.duplicated().sum()

np.int64(0)

In [45]:
customer_sales = sales[
    sales["CustomerID"].notna()
].copy()

In [46]:
print("Sales dataset:", sales.shape)
print("Customer analytics dataset:", customer_sales.shape)

Sales dataset: (1029609, 16)
Customer analytics dataset: (793609, 16)


In [47]:
cancelled = df[
    df["IsCancelled"] |
    (df["Quantity"] < 0)
].copy()

In [48]:
cancelled.shape

(22951, 16)

In [49]:
quality_summary = pd.DataFrame({
    "Metric": [
        "Original Rows",
        "Completed Sales Rows",
        "Customer Analytics Rows",
        "Cancelled / Negative Quantity Rows",
        "Exact Duplicates in Original Data",
        "Missing Customer IDs"
    ],
    "Count": [
        len(df),
        len(sales),
        len(customer_sales),
        len(cancelled),
        duplicate_count,
        df["CustomerID"].isna().sum()
    ]
})

quality_summary

,Metric,Count
0,Original Rows,1067371
1,Completed Sales Rows,1029609
2,Customer Analytics Rows,793609
3,Cancelled / Negative Quantity Rows,22951
4,Exact Duplicates in Original Data,12133
5,Missing Customer IDs,243007


In [50]:
sales.info()


<class 'pandas.core.frame.DataFrame'>
Index: 1029609 entries, 0 to 1067370
Data columns (total 16 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   InvoiceNo    1029609 non-null  object        
 1   StockCode    1029609 non-null  object        
 2   Description  1029609 non-null  object        
 3   Quantity     1029609 non-null  int64         
 4   InvoiceDate  1029609 non-null  datetime64[ns]
 5   UnitPrice    1029609 non-null  float64       
 6   CustomerID   793609 non-null   float64       
 7   Country      1029609 non-null  object        
 8   SourceYear   1029609 non-null  object        
 9   IsCancelled  1029609 non-null  bool          
 10  Year         1029609 non-null  int32         
 11  Month        1029609 non-null  int32         
 12  YearMonth    1029609 non-null  object        
 13  DayOfWeek    1029609 non-null  object        
 14  Hour         1029609 non-null  int32         
 15  Revenue      1029609

In [51]:
sales.describe(

)

,Quantity,InvoiceDate,UnitPrice,CustomerID,Year,Month,Hour,Revenue
count,"1,029,609.00",1029609,"1,029,609.00","793,609.00","1,029,609.00","1,029,609.00","1,029,609.00","1,029,609.00"
mean,11.06,2011-01-03 19:35:55.374282752,4.09,"15,325.07","2,010.43",7.50,13.03,20.31
min,1.00,2009-12-01 07:45:00,0.00,"12,346.00","2,009.00",1.00,6.00,0.00
25%,1.00,2010-07-12 10:54:00,1.25,"13,975.00","2,010.00",5.00,11.00,4.13
50%,3.00,2010-12-07 18:36:00,2.10,"15,255.00","2,010.00",8.00,13.00,10.08
75%,12.00,2011-07-24 12:05:00,4.15,"16,797.00","2,011.00",11.00,15.00,17.70
max,"80,995.00",2011-12-09 12:50:00,"25,111.09","18,287.00","2,011.00",12.00,20.00,"168,469.60"
std,127.24,NaN,51.75,"1,697.40",0.57,3.53,2.43,204.29


In [52]:
sales.head(

)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,SourceYear,IsCancelled,Year,Month,YearMonth,DayOfWeek,Hour,Revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,2009-2010,False,2009,12,2009-12,Tuesday,7,30.00


In [53]:
sales.to_csv(
    "/content/cleaned_sales.csv",
    index=False
)

In [54]:
customer_sales.to_csv(
    "/content/customer_sales.csv",
    index=False
)

In [55]:
cancelled.to_csv(
    "/content/cancelled_transactions.csv",
    index=False
)

In [56]:
sales.sample(
    n=min(10000, len(sales)),
    random_state=42
).to_csv(
    "/content/cleaned_sales_sample.csv",
    index=False
)